# 實作情緒分析(Sentiment Analysis)

## 載入IMDB資料集 

In [1]:
from __future__ import annotations

import time
from collections import Counter, OrderedDict
from typing import Callable, Iterable, Iterator, cast, overload

import torch
from datasets import Dataset as HFDataset
from datasets import load_dataset
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, random_split
from text_utils import get_tokenizer
from text_utils import vocab as build_vocab

In [2]:
# torchtext.datasets.IMDB 已棄用，改用 Hugging Face datasets 載入 IMDB 資料集
# 標籤對照：HF 資料集標籤為 0(neg)/1(pos)，torchtext 版本為 'neg'/'pos' 字串，故轉換以維持相容
IMDBIterator = Iterator[tuple[str, str]]


@overload
def IMDB(split: str) -> IMDBIterator: ...
@overload
def IMDB(split: tuple[str, str] = ('train', 'test')) -> tuple[IMDBIterator, IMDBIterator]: ...
def IMDB(
    split: str | tuple[str, str] = ('train', 'test'),
) -> IMDBIterator | tuple[IMDBIterator, IMDBIterator]:
    single = isinstance(split, str)
    splits = (split,) if single else split
    results = []
    for s in splits:
        ds = cast(HFDataset, load_dataset('stanfordnlp/imdb', split=s))
        results.append(
            (('neg' if cast(dict, example)['label'] == 0 else 'pos'), cast(dict, example)['text']) for example in ds
        )
    return results[0] if single else tuple(results)


imdb = IMDB(split='train')

type(imdb)

generator

In [3]:
train_iter = iter(IMDB(split='train'))

In [4]:
# 取得下一筆資料
data = next(train_iter)
data

('neg',
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 判斷GPU是否存在

In [5]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'mps'

## 詞彙表處理

In [6]:
# 分詞
tokenizer = get_tokenizer('basic_english')


# 建立 Generator 函數
def yield_tokens(data_iter: Iterable[tuple[str, str]]) -> Iterator[list[str]]:
    for _, text in data_iter:
        yield tokenizer(text)


# 由 train_iter 建立詞彙字典(統計詞頻，再依出現次數降冪排列)
counter = Counter()
for tokens in yield_tokens(train_iter):
    counter.update(tokens)
sorted_by_freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = build_vocab(ordered_dict, specials=["<unk>"])

# 設定預設的索引值
vocab.set_default_index(vocab["<unk>"])

In [7]:
# 測試詞彙字典，取得單字的索引值
vocab.lookup_indices(['here', 'is', 'an', 'example'])

[131, 9, 40, 464]

## 參數設定

In [8]:
EPOCHS = 10  # 訓練週期數
LR = 5  # 學習率
BATCH_SIZE = 64  # 訓練批量
# 取得標註個數
num_class = len(set([label for (label, text) in imdb]))
vocab_size = len(vocab)
emsize = 64
hidden_dim = 32

In [9]:
num_class

2

## 定義資料轉換函數

In [10]:
text_pipeline = lambda x: vocab.lookup_indices(tokenizer(x))  # 分詞、取得單字的索引值
label_pipeline = lambda x: 0 if x == 'neg' else 1  # 換成索引值

In [11]:
# 測試資料轉換
print(text_pipeline('here is an example'))
label_pipeline('pos')

[131, 9, 40, 464]


1

## 建立模型

In [12]:
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_class: int) -> None:
        super().__init__()
        # sparse=True 會導致新版 PyTorch 的 clip_grad_norm_ 在 CPU 上失敗(不支援稀疏梯度)，故改為 False
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.rnn = nn.RNN(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_class)
        self.init_weights()

    def init_weights(self) -> None:
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(text, offsets)
        rnn_out, h_out = self.rnn(embedded)
        return self.fc(rnn_out)


model = TextClassificationModel(vocab_size, emsize, num_class).to(device)

## 定義訓練及評估函數

In [13]:
# 訓練函數
def train(dataloader: DataLoader) -> None:
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(
                f'| epoch {epoch:3d} | {idx:5d}/{len(dataloader):5d} batches '
                f'| accuracy {total_acc / total_count:8.3f}'
            )
            total_acc, total_count = 0, 0
            start_time = time.time()


# 評估函數
def evaluate(dataloader: DataLoader) -> float:
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label = model(text, offsets)
            loss = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

## 建立DataLoader，逐批訓練

In [14]:
# torchtext.data.functional.to_map_style_dataset 已棄用，改用 list 即可取得支援 __len__/__getitem__ 的資料集
def to_map_style_dataset(data_iter: Iterable[tuple[str, str]]) -> list[tuple[str, str]]:
    return list(data_iter)


# 批次處理
def collate_batch(batch: Iterable[tuple[str, str]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    label_list, text_list, offsets = [], [], [0]
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))  # 設定每筆資料的起始位置
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)  # 每筆資料的起始位置累加
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)


train_iter, test_iter = IMDB()
# 轉換為 DataSet
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)
# 資料切割，95% 作為訓練資料
num_train = int(len(train_dataset) * 0.95)
split_train_, split_valid_ = random_split(
    cast(Dataset[tuple[str, str]], train_dataset), [num_train, len(train_dataset) - num_train]
)

# 建立DataLoader
train_dataloader = DataLoader(split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(
    cast(Dataset[tuple[str, str]], test_dataset), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

## 模型訓練

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.1)

total_accu = None
for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    train(train_dataloader)
    accu_val = evaluate(valid_dataloader)
    if total_accu is not None and total_accu > accu_val:
        scheduler.step()
    else:
        total_accu = accu_val
    print('-' * 59)
    print(
        f'| end of epoch {epoch:3d} | time: {time.time() - epoch_start_time:5.2f}s | '
        f'valid accuracy {accu_val:8.3f} '
    )
    print('-' * 59)

-----------------------------------------------------------
| end of epoch   1 | time: 66.89s | valid accuracy    0.632 
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   2 | time: 21.29s | valid accuracy    0.789 
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   3 | time: 38.91s | valid accuracy    0.719 
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   4 | time: 30.54s | valid accuracy    0.833 
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   5 | time: 16.98s | valid accuracy    0.834 
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   6 | time: 31.46s |

## 模型評估

In [16]:
print(f'測試資料準確度: {evaluate(test_dataloader):.3f}')

測試資料準確度: 0.839


## 測試新資料

In [17]:
# 預測
label = {0: '負面', 1: '正面'}


def predict(text: str, text_pipeline: Callable[[str], list[int]]) -> int:
    with torch.no_grad():
        text_tensor = torch.tensor(text_pipeline(text)).to(device)
        output = model(text_tensor, torch.tensor([0]).to(device))
        return output.argmax(1).item()


# 測試資料
my_test = open('nlp_data/imdb_1.txt', encoding='utf8').read()
print(label[predict(data[1], text_pipeline)])

負面


In [18]:
imdb_iterator = iter(IMDB(split='train'))
label_rev = {'neg': 0, 'pos': 1}
acc = 0
for i in range(20000):
    data = next(imdb_iterator)
    acc += 1 if label_rev[data[0]] == predict(data[1], text_pipeline) else 0
print(acc)

17237
